In [1]:
import os
import torch
from huggingface_hub import hf_hub_download
import sys
# # Add the parent directory of 'pipeline_flux' to the Python path
# print(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
# sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..', '..')))
from pipeline_flux import FluxPipeline
from transformer_flux import FluxTransformer2DModel

In [2]:
bfl_repo="black-forest-labs/FLUX.1-dev"
device = torch.device('cuda:3')
dtype = torch.bfloat16
transformer = FluxTransformer2DModel.from_pretrained(bfl_repo, subfolder="transformer", torch_dtype=dtype)
pipe = FluxPipeline.from_pretrained(bfl_repo, transformer=transformer, torch_dtype=dtype)
pipe.scheduler.config.use_dynamic_shifting = False
pipe.scheduler.config.time_shift = 10
pipe = pipe.to(device)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [21]:
if not os.path.exists('ckpt/urae_2k_adapter.safetensors'):
    hf_hub_download(repo_id="Huage001/URAE", filename='urae_2k_adapter.safetensors', local_dir='ckpt', local_dir_use_symlinks=False)
pipe.load_lora_weights("ckpt/urae_2k_adapter.safetensors")
pipe = pipe.to(device)

/mnt/share/Luigi/anaconda3/envs/URAE/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:167: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
pipe.load_lora_weights("ckpt/original_URAE/checkpoint-2000/pytorch_lora_weights.safetensors")
pipe = pipe.to(device)

/mnt/share/Luigi/anaconda3/envs/URAE/lib/python3.12/site-packages/peft/tuners/tuners_utils.py:167: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [ ]:
prompt = "A serene woman in a flowing azure dress, gracefully perched on a sunlit cliff overlooking a tranquil sea, her hair gently tousled by the breeze. The scene is infused with a sense of peace, evoking a dreamlike atmosphere, reminiscent of Impressionist paintings."
height = 2048
width = 2048
image = pipe(
    prompt,
    height=height,
    width=width,
    guidance_scale=3.5,
    num_inference_steps=28,
    max_sequence_length=512,
    generator=torch.manual_seed(8888),
    ntk_factor=10,
    proportional_attention=True
).images[0]
image

In [22]:
import os
import hpsv2
from tqdm.notebook import tqdm
# Get benchmark prompts (<style> = all, anime, concept-art, paintings, photo)
all_prompts = hpsv2.benchmark_prompts('all') 
height = 2048
width = 2048
# Iterate over the benchmark prompts to generate images
for style, prompts in tqdm(all_prompts.items()):
    # Create the directory if it doesn't exist
    output_dir = os.path.join("output/hpsv2_urae_original", style)
    os.makedirs(output_dir, exist_ok=True)
    for idx, prompt in tqdm(enumerate(prompts), total=len(prompts)):
        # image = TextToImageModel(prompt)
        image = pipe(
            prompt,
            height=height,
            width=width,
            guidance_scale=3.5,
            num_inference_steps=28,
            max_sequence_length=512,
            generator=torch.manual_seed(8888),
            ntk_factor=10,
            proportional_attention=True
        ).images[0]
        # TextToImageModel is the model you want to evaluate
        image.save(os.path.join("output/hpsv2_urae_original", style, f"{idx:05d}.jpg")) 
        # <image_path> is the folder path to store generated images, as the input of hpsv2.evaluate().


  0%|          | 0/4 [00:00<?, ?it/s]

  0%|          | 0/800 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

...

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

  0%|          | 0/28 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [15]:
import hpsv2

# <image_path> is the same as <image_path> in the prevoius part.
# <hps_version> is the version of HPS model, it can be v2 or v2.1. Default to v2.
hpsv2.evaluate("output/hpsv2_urae_trained_by_me",) 


HPS_v2.1_compressed.pt:   0%|          | 0.00/1.97G [00:00<?, ?B/s]

Loading model ...
Loading model successfully!
missing image for prompt: A girl in school uniform standing in the city.
missing image for prompt: Image of xqc with a distinctive underbite and big, long nose.
missing image for prompt: A goth anime woman with a symmetrical and attractive face in a black and white watercolor headshot art on ArtStation.
missing image for prompt: Peter Griffin taking a selfie in a park on an iPhone 12 Pro.
missing image for prompt: A helmet-wearing monkey skating.
missing image for prompt: Anime oil painting of Rem from Re Zero.
missing image for prompt: Pinup girl in the style of Jab Comix.
missing image for prompt: A photo of Big Chungus from Looney Tunes.
missing image for prompt: Frontal portrait of anime girl with pink hair wearing white t-shirt and smiling.
missing image for prompt: Tom and Jerry are featured in Iron Maiden's album Live After Death in place of their usual mascot, Eddie.
missing image for prompt: A kangaroo in an orange hoodie and blue 

/mnt/share/Luigi/anaconda3/envs/URAE/lib/python3.12/site-packages/hpsv2/evaluation.py:138: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


-----------benchmark score ---------------- 
hpsv2_urae_original anime           30.57 	 0.0000
hpsv2_urae_original Average         30.57 	


In [ ]:
import hpsv2

hpsv2.evaluate("output/hpsv2_urae_original") 

Loading model ...
Loading model successfully!
missing image for prompt: A girl in school uniform standing in the city.
missing image for prompt: Image of xqc with a distinctive underbite and big, long nose.
missing image for prompt: A goth anime woman with a symmetrical and attractive face in a black and white watercolor headshot art on ArtStation.
missing image for prompt: Peter Griffin taking a selfie in a park on an iPhone 12 Pro.
missing image for prompt: A helmet-wearing monkey skating.
missing image for prompt: Anime oil painting of Rem from Re Zero.
missing image for prompt: Pinup girl in the style of Jab Comix.
missing image for prompt: A photo of Big Chungus from Looney Tunes.
missing image for prompt: Frontal portrait of anime girl with pink hair wearing white t-shirt and smiling.
missing image for prompt: Tom and Jerry are featured in Iron Maiden's album Live After Death in place of their usual mascot, Eddie.
missing image for prompt: A kangaroo in an orange hoodie and blue 

/mnt/share/Luigi/anaconda3/envs/URAE/lib/python3.12/site-packages/hpsv2/evaluation.py:138: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


: 